# Phase 2: Environment Setup

**Goal:** confirm GR00T N1.7 (action/System1) and Cosmos-Reason2-2B (reasoning proxy/System2) both load and run for inference on Colab Pro, pull a small AgiBot World subset (10-20 trajectories), and verify the data format end-to-end. Nothing here trains or fine-tunes anything — inference only.

**Runtime:** Runtime > Change runtime type > GPU > A100 (Colab Pro/Pro+). If A100 is unavailable, L4 is the fallback; both should fit GR00T N1.7 (~16GB+) and Cosmos-Reason2-2B (~4-6GB) run sequentially.

**Checkpointing:** everything that takes real time (downloads, model loads, verification results) writes a status marker to Google Drive so a disconnected session can resume without redoing completed steps. Re-run all cells top to bottom after a reconnect — completed steps will just print "already done" and skip.

**If any cell errors: stop, copy the full error output, and paste it back.** Do not try to fix it yourself — several API details below (exact HF repo IDs, exact GR00T loading calls) are marked `VERIFY` because they could not be confirmed without running code, and the error message is what tells us what to adjust.

## 0. GPU check, Drive mount, checkpoint scaffolding

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time

PROJECT_DIR = '/content/drive/MyDrive/humanoid-vla-eval'
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
DATA_DIR = os.path.join(PROJECT_DIR, 'data', 'agibot_subset')
STATUS_PATH = os.path.join(CKPT_DIR, 'phase2_status.json')

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

def load_status():
    if os.path.exists(STATUS_PATH):
        with open(STATUS_PATH) as f:
            return json.load(f)
    return {}

def save_status(key, value):
    status = load_status()
    status[key] = value
    status['_last_updated'] = time.strftime('%Y-%m-%d %H:%M:%S')
    with open(STATUS_PATH, 'w') as f:
        json.dump(status, f, indent=2)
    return status

def is_done(key):
    return load_status().get(key) == 'done'

print('Current status:', load_status())

## 1. Install dependencies

In [ ]:
import subprocess, platform

def run(cmd):
    """Run a shell command and raise loudly on failure — a bare `!cmd` in Colab does
    NOT raise on nonzero exit, which is why the previous version of this cell silently
    marked a failed install as done."""
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode}): {cmd}')

if not is_done('deps_installed'):
    py_ver = platform.python_version()
    print(f'Python version: {py_ver}')
    if not py_ver.startswith('3.12'):
        print('WARNING: Isaac-GR00T requires Python 3.12.x (pyproject.toml: >=3.12,<3.13). '
              'If the install below fails with a python-version error, stop and paste it back.')

    if not os.path.exists('/content/Isaac-GR00T'):
        run('git clone https://github.com/NVIDIA/Isaac-GR00T.git /content/Isaac-GR00T')

    # NVIDIA's pyproject.toml pins flash-attn to a prebuilt wheel URL via
    # [tool.uv.sources], which only `uv` resolves. Plain `pip install -e .` tries to
    # compile flash-attn from source instead and fails on Colab (no matching
    # build toolchain / very long compile). `uv pip install --system` reads
    # [tool.uv.sources] correctly and pulls the prebuilt wheel instead.
    run('pip install -q -U uv')
    run('uv pip install --system -e /content/Isaac-GR00T')
    run('uv pip install --system accelerate')

    save_status('deps_installed', 'done')
    print('Dependencies installed.')
else:
    print('Dependencies already installed (per checkpoint) — skipping. Delete the status key to force reinstall.')

## 2. Hugging Face auth

GR00T checkpoints, Cosmos-Reason2, and AgiBot World are all gated on Hugging Face. You need to:
1. Accept the license/terms on each model/dataset page while logged into your HF account:
   - https://huggingface.co/nvidia/GR00T-N1.7-3B
   - https://huggingface.co/nvidia/Cosmos-Reason2-2B
   - https://huggingface.co/datasets/agibot-world/AgiBotWorld-Beta
2. Generate a read-access token at https://huggingface.co/settings/tokens
3. Paste it below when prompted (input is hidden).

In [ ]:
from huggingface_hub import login
login()

## 3. Load GR00T N1.7 (System 1 / action model)

In [ ]:
import torch

GR00T_CHECKPOINT = 'nvidia/GR00T-N1.7-3B'  # VERIFY exact repo id if this 404s

gr00t_policy = None
gr00t_error = None
try:
    from gr00t.model.policy import Gr00tPolicy
    from gr00t.data.embodiment_tags import EmbodimentTag

    print('Available embodiment tags:', [t for t in dir(EmbodimentTag) if not t.startswith('_')])

    gr00t_policy = Gr00tPolicy(
        model_path=GR00T_CHECKPOINT,
        embodiment_tag=EmbodimentTag.AGIBOT_GENIE1,
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )
    print('GR00T N1.7 loaded successfully with EmbodimentTag.AGIBOT_GENIE1.')
    save_status('gr00t_loaded', 'done')
except Exception as e:
    gr00t_error = e
    print('GR00T load FAILED — copy this full traceback back to Claude:')
    raise

## 4. Load Cosmos-Reason2-2B (System 2 reasoning proxy)

This is run as a **standalone natural-language chain-of-thought reasoner**, not as part of GR00T's internal pipeline. It is a proxy for GR00T's true (inaccessible, latent-only) internal reasoning representation — same backbone lineage, but not literally the same weights or the same computation GR00T performs internally. This must stay labeled as a proxy everywhere downstream.

In [ ]:
COSMOS_REASON2_CHECKPOINT = 'nvidia/Cosmos-Reason2-2B'  # VERIFY exact repo id if this 404s

from transformers import AutoModelForCausalLM, AutoProcessor

cosmos_model = None
cosmos_processor = None
try:
    cosmos_processor = AutoProcessor.from_pretrained(COSMOS_REASON2_CHECKPOINT, trust_remote_code=True)
    cosmos_model = AutoModelForCausalLM.from_pretrained(
        COSMOS_REASON2_CHECKPOINT,
        torch_dtype=torch.bfloat16,
        device_map='cuda' if torch.cuda.is_available() else 'cpu',
        trust_remote_code=True,
    )
    print('Cosmos-Reason2-2B loaded successfully.')
    save_status('cosmos_loaded', 'done')
except Exception as e:
    print('Cosmos-Reason2 load FAILED — copy this full traceback back to Claude:')
    raise

## 5. Discover AgiBot World structure and pull a small subset

We list the dataset's actual repo structure live (rather than hardcoding task folder names that could be stale) and sparse-checkout only a small number of task folders — targeting roughly 10-20 trajectories, not the full 92K/1M-trajectory releases.

In [ ]:
from huggingface_hub import HfApi

AGIBOT_REPO = 'agibot-world/AgiBotWorld-Beta'  # VERIFY: Alpha may be easier for a first pass (smaller, curated)

api = HfApi()
files = api.list_repo_files(AGIBOT_REPO, repo_type='dataset')
print(f'{len(files)} files in repo. First 40:')
for f in files[:40]:
    print(' ', f)

# Identify distinct task_ folders present at the top level
task_ids = sorted({f.split('/')[0] for f in files if f.startswith('task_') or '/task_' in f})
print(f'\nFound {len(task_ids)} task folders. First 10: {task_ids[:10]}')

In [ ]:
from huggingface_hub import snapshot_download

N_TASKS_TO_PULL = 3  # small enough to land well under 20 trajectories total; adjust after seeing per-task trajectory counts
selected_tasks = task_ids[:N_TASKS_TO_PULL]
print('Pulling tasks:', selected_tasks)

if not is_done('agibot_subset_downloaded'):
    allow_patterns = [f'{t}/**' for t in selected_tasks] + [f'*{t}*' for t in selected_tasks]
    snapshot_download(
        repo_id=AGIBOT_REPO,
        repo_type='dataset',
        allow_patterns=allow_patterns,
        local_dir=DATA_DIR,
    )
    save_status('agibot_subset_downloaded', 'done')
    save_status('agibot_selected_tasks', selected_tasks)
else:
    print('AgiBot subset already downloaded — skipping.')

!du -sh {DATA_DIR}

## 6. Parse one trajectory's format

Confirm we can read: (a) the language instruction and segment-level sub-goal annotations, (b) at least one visual frame, (c) the action array — before building anything automated on top.

In [ ]:
import glob

# List what actually landed on disk so we can see the real structure
downloaded = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*'), recursive=True))
print(f'{len(downloaded)} paths downloaded. Sample:')
for p in downloaded[:60]:
    print(' ', p.replace(DATA_DIR + '/', ''))

In [ ]:
# VERIFY: this cell is intentionally left for us to fill in together once cell above
# shows the real on-disk layout (AgiBot's exact json/video/action file naming).
# Paste the output of the previous cell back and we'll finish this cell together
# rather than guessing a schema that may not match what actually downloaded.
print('Waiting on real directory listing before writing the parser.')

## 7. End-to-end smoke test (one frame, both models)

Deferred until cell 6's parser is finalized against the real data layout.

In [ ]:
print('Deferred — depends on Section 6.')

## 8. Status summary

In [ ]:
print(json.dumps(load_status(), indent=2))